# A coastal boundary for the synthetic valley

The synthetic valley drains south. Its southern edge is closed except where the
river leaves the domain, so the recharge that is not pumped, evaporated, or
taken by the seepage drains has to reach the river. Put the sea along that edge
instead and the valley becomes a coastal aquifer, with the river and the coast
competing for the same water.

## What this notebook covers

Add a **general-head boundary** (**GHB**) at sea level along the southern row of
the valley, run the model with and without it, and measure what the coast takes
from the river. Then repeat the comparison on the advanced valley, where the
river is a streamflow-routing (**SFR**) network and the lake, unsaturated zone,
and water mover are all active.

By the end of this notebook you will be able to:

- build a GHB along a model edge from the grid and conductivity arrays,
- say what the `ELEVATION` and `CONCENTRATION` auxiliary variables are for,
- compare two simulations through their budget files and observation output, and
- read the discharge split between a stream and a coast from those budgets.

A GHB is a head-dependent boundary: it moves water into or out of a cell in
proportion to the difference between a fixed boundary head and the simulated
head in the cell, scaled by a **conductance** that carries the units of the
sediment between them. The sea is the natural thing to represent this way,
because its level is set by something far larger than the aquifer.

Import the packages this notebook uses. `mf6_notebook_helpers` holds the shared
synthetic-valley setup: the coastal boundary builder used below and the
functions that show results in meters or feet. The models are in meters; set
the environment variable `MF6_TRAINING_UNITS` to `feet` to see results in feet.
`display_ticks()` puts the map axes on round values in those units.

In [ ]:
%matplotlib inline

import pathlib as pl
import shutil

import flopy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mf6_notebook_helpers import (
    coastal_ghb_data,
    display_ticks,
    find_mf6_libraries,
    length_name,
    to_display,
    unit,
)

Locate the MODFLOW 6 executable in the active environment.

In [ ]:
_, mf6_exe = find_mf6_libraries()
print(f"executable: {mf6_exe.name}")

## Look at the valley before the sea

Load the base model. It is a grid of 5 layers, 40 rows, and 25 columns of
152.4-m (500-ft) cells, in meters and days. Layers 1 and 2 are the upper
aquifer and layers 4 and 5 the lower aquifer; layer 3 is the clay where there is
clay and has no thickness elsewhere. Water enters as recharge (**RCH**) and
leaves through evaporation from the lake (**EVT**), a river (**RIV**) running
down column 9, seepage drains (**DRN**) that take groundwater out where the
water table reaches the land surface, two production wells, and a prediction
well that starts pumping in stress period 12. The lake in the north is layer 1
cells of very high hydraulic conductivity rather than a lake package.

In [ ]:
data_root = pl.Path("../data/synthetic-valley")
base_ws = data_root / "synthetic-valley-base-annual"

sim = flopy.mf6.MFSimulation.load(sim_ws=str(base_ws), verbosity_level=0)
gwf = sim.get_model()
nlay, nrow, ncol = gwf.dis.nlay.data, gwf.dis.nrow.data, gwf.dis.ncol.data
south = nrow - 1  # zero-based index of the southern row
outlet_col = 8  # zero-based column the river leaves the domain in

top_south = to_display(gwf.dis.top.array[south])
print(f"grid:            {nlay} layers, {nrow} rows, {ncol} columns")
print(f"stress periods:  {sim.tdis.nper.data}")
print(
    "land surface along the southern row: "
    f"{top_south.min():.1f} to {top_south.max():.1f} {unit(plain=True)}"
)

Map the valley to see where the sea will go. Plot the river cells with
`.plot_bc()`, mark the southern row, and mark the cell the river leaves through.
The figure is drawn inside `flopy.plot.styles.USGSMap()`, which supplies the
fonts and tick geometry used throughout the training material.

In [ ]:
xc, yc = gwf.modelgrid.xcellcenters, gwf.modelgrid.ycellcenters

with flopy.plot.styles.USGSMap():
    fig, ax = plt.subplots(figsize=(4.5, 7), layout="constrained")
    mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
    mm.plot_grid(lw=0.2, color="0.85")
    mm.plot_bc("RIV", color="tab:cyan")
    coast_cols = [j for j in range(ncol) if j != outlet_col]
    ax.plot(
        xc[south, coast_cols],
        yc[south, coast_cols],
        "s",
        color="tab:blue",
        ms=6,
        label="coast",
    )
    ax.plot(
        xc[south, outlet_col],
        yc[south, outlet_col],
        "s",
        color="tab:red",
        ms=6,
        label="river outlet",
    )
    ax.plot([], [], "s", color="tab:cyan", label="river")
    ax.legend(loc="upper right", fontsize=8)
    # label the axes in display units; the model grid stays in meters
    display_ticks(ax.xaxis)
    display_ticks(ax.yaxis)
    ax.set_xlabel(f"x, in {length_name()}")
    ax.set_ylabel(f"y, in {length_name()}")
    ax.set_title("Southern edge of the synthetic valley")

**What to look for.** The river runs down column 9 from row 23 and leaves
through the red cell in the southern row. Every other cell in that row (blue) is
closed today and becomes part of the coast. Land surface along the row is 2.1 to
4.2 m (6.9 to 13.7 ft), so a sea at 0 m sits below the valley floor.

## Build the coastal boundary

A GHB is a list package, so each entry is a tuple. The valley models use
boundary names and two auxiliary variables, which puts the fields in this order:

```python
# ((layer, row, column), bhead, cond, elevation, concentration, boundname)
((4, 39, 24), 0.0, 2129.6, -38.37, 35.0, "coast")
```

The model input is in meters and days. `bhead` is sea level, the same 0.0 m in
every layer. MODFLOW 6 solves in hydraulic head, and a column of seawater whose
surface sits at sea level has that same head at every depth, so no depth
correction belongs in the input.

`ELEVATION` is the middle of the submerged part of the cell face and
`CONCENTRATION` is the salinity of seawater, 35 kg/m³. Both are inert in this
notebook: they matter only once the Buoyancy (**BUY**) package is active and
density varies, which happens in
[`mf6-coastal-density`](mf6-coastal-density.ipynb). Carrying them from the start
means the same boundary serves both models.

Conductance is the one value that has to be computed per cell. Take it as the
conductance of the cell itself across its outer face - hydraulic conductivity
times the face area, divided by the distance from the cell center to that face.
`coastal_ghb_data()` does this for every active cell in the row, skipping the
outlet column so the river keeps the only surface outflow there. There is no
clay under the southern row, so layer 3 has no thickness there and gets no
boundary.

In [ ]:
# exercise: build the GHB data for the southern row with coastal_ghb_data()
# your code here

cond = np.array([to_display(c[2], "conductance") for c in ghb_spd])
print(f"boundary cells:  {len(ghb_spd)}")
print(f"auxiliary:       {', '.join(ghb_aux)}")
print(
    f"conductance:     {cond.min():,.0f} to {cond.max():,.0f} "
    f"{unit('conductance', plain=True)}"
)
print(f"total:           {cond.sum():,.0f} {unit('conductance', plain=True)}")
by_layer = {}
for (k, _, _), c in zip((rec[0] for rec in ghb_spd), cond):
    by_layer[k + 1] = by_layer.get(k + 1, 0.0) + c
for k, c in by_layer.items():
    print(f"  layer {k}: {c:12,.0f} {unit('conductance', plain=True)}")
riv_cond = to_display(
    gwf.get_package("riv-1").stress_period_data.get_data(0)["cond"].sum(),
    "conductance",
)
print(f"river:           {riv_cond:,.0f} {unit('conductance', plain=True)}")
cellid, bhead, cond_m, elev, conc, name = ghb_spd[-1]
print(
    f"last entry (model input, in meters): "
    f"({cellid}, {bhead:.1f}, {cond_m:.1f}, {elev:.2f}, {conc:.1f}, {name!r})"
)

**What to look for.** The 96 boundary cells, layers 1, 2, 4, and 5 of the 24
coastal columns, carry 233,047 m²/d (2,508,497 ft²/d) of conductance between
them. Layers 4 and 5 hold 89 percent of it because they are the thickest, 38 and
27 m (125 and 89 ft) on average along the row compared to 5 and 3 m (16 and 10
ft) for layers 1 and 2. For comparison, the river's conductance totals 167,225
m²/d (1,800,000 ft²/d), so the coast is the better-connected of the two sinks.

## Run the valley with and without the sea

Copy the shipped model twice. One copy is left alone, and the other gets the GHB
package built above. Everything else - recharge, evaporation, the river, the
seepage drains, the wells - is identical, so any difference between the two runs
belongs to the coast.

In [ ]:
model_root = pl.Path("models")
no_coast_ws = model_root / "coastal-ghb-no-coast"
coastal_ws = model_root / "coastal-ghb-base"

for ws in (no_coast_ws, coastal_ws):
    if ws.exists():
        shutil.rmtree(ws)
    shutil.copytree(base_ws, ws)

sim = flopy.mf6.MFSimulation.load(sim_ws=str(coastal_ws), verbosity_level=0)
gwf_coastal = sim.get_model()
flopy.mf6.ModflowGwfghb(
    gwf_coastal,
    pname="ghb-1",
    auxiliary=ghb_aux,
    boundnames=True,
    stress_period_data={0: ghb_spd},
)
sim.write_simulation(silent=True)

for ws in (no_coast_ws, coastal_ws):
    success, buff = flopy.run_model(
        exe_name=str(mf6_exe), namefile=None, model_ws=str(ws), silent=True
    )
    if not success:
        raise RuntimeError("\n".join(buff[-15:]))
print("both simulations converged")

Read the last stress period of each budget file. `sv-budget.csv` carries one
column per package for flow in and flow out, so the net discharge to a boundary
is the difference between them. The seepage drains are read as well, since they
are the third way groundwater leaves the valley.

In [ ]:
def net_flow(ws, term):
    """Net flow out of the aquifer to one package in the last stress period."""
    budget = pd.read_csv(pl.Path(ws) / "sv-budget.csv").iloc[-1]
    return to_display(budget[f"{term}_OUT"] - budget[f"{term}_IN"], "flow")


flow_unit = unit("flow", plain=True)
river_before = net_flow(no_coast_ws, "RIV(RIV-1)")
river_after = net_flow(coastal_ws, "RIV(RIV-1)")
drains_before = net_flow(no_coast_ws, "DRN(DRN-1)")
drains_after = net_flow(coastal_ws, "DRN(DRN-1)")
coast_after = net_flow(coastal_ws, "GHB(GHB-1)")

print(f"river, no coast:  {river_before:12,.0f} {flow_unit}")
print(
    f"river, coastal:   {river_after:12,.0f} {flow_unit} "
    f"({100 * river_after / river_before:.0f} percent retained)"
)
print(f"drains, no coast: {drains_before:12,.0f} {flow_unit}")
print(f"drains, coastal:  {drains_after:12,.0f} {flow_unit}")
print(f"coast, coastal:   {coast_after:12,.0f} {flow_unit}")
print(
    "coast share:      "
    f"{100 * coast_after / (coast_after + river_after + drains_after):12.0f} "
    "percent of the discharge to the coast, river, and drains"
)

Plot what the sea does to the water table. Panel A follows the head along the
southern row in both runs, and panel B maps how much the head falls when the
coast is added.

In [ ]:
head_before = to_display(flopy.utils.HeadFile(no_coast_ws / "sv.hds").get_alldata()[-1])
head_after = to_display(flopy.utils.HeadFile(coastal_ws / "sv.hds").get_alldata()[-1])
drawdown = head_before[0] - head_after[0]

with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic([["A", "B"]], figsize=(9.5, 6), layout="constrained")

    ax = axd["A"]
    ax.plot(
        np.arange(1, ncol + 1),
        head_before[0, south],
        "o-",
        color="tab:gray",
        label="no coast",
    )
    ax.plot(
        np.arange(1, ncol + 1),
        head_after[0, south],
        "o-",
        color="tab:blue",
        label="coastal",
    )
    ax.axhline(0.0, lw=0.8, ls=":", color="k", label="sea level")
    ax.set_xlabel("column")
    ax.set_ylabel(f"head, in {length_name()}")
    ax.legend(fontsize=8)
    ax.set_title("A. Head along the southern row")

    ax = axd["B"]
    mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
    cb = mm.plot_array(drawdown, cmap="magma_r")
    mm.plot_bc("RIV", color="tab:cyan")
    fig.colorbar(cb, ax=ax, shrink=0.5, label=f"head decline, in {length_name()}")
    display_ticks(ax.xaxis)
    display_ticks(ax.yaxis)
    ax.set_xlabel(f"x, in {length_name()}")
    ax.set_ylabel(f"y, in {length_name()}")
    ax.set_title("B. Head decline in layer 1")

active = gwf.dis.idomain.array[0] > 0
print(
    f"southern row, no coast: {np.nanmin(head_before[0, south]):.2f} to "
    f"{np.nanmax(head_before[0, south]):.2f} {unit(plain=True)}"
)
print(
    f"southern row, coastal:  {np.nanmin(head_after[0, south]):.2f} to "
    f"{np.nanmax(head_after[0, south]):.2f} {unit(plain=True)}"
)
print(
    f"layer 1 decline: largest {drawdown[active].max():.2f}, "
    f"mean {drawdown[active].mean():.2f} {unit(plain=True)}"
)

**What to look for.** In panel A the head along the southern row falls from
0.19-2.47 m (0.63-8.10 ft) to 0.03-0.11 m (0.11-0.34 ft), which is the coast
holding that edge at sea level. Panel B shows the decline reaching back through
the valley: it is 2.37 m (7.76 ft) at most, in the southeast corner, and
averages 0.46 m (1.51 ft) over layer 1. It is smallest along the river, which
holds the water table there, and is 0.28 m (0.91 ft) under the lake.

The budget above is the result to keep: the coast takes 12,713 m³/d
(448,946 ft³/d) and the river keeps 24,718 m³/d (872,893 ft³/d) of the
37,510 m³/d (1,324,666 ft³/d) it carried before, which is two thirds of it. The
seepage drains carry nothing in the last stress period of either run; the water
table is below them by then. The coast has more conductance than the river but
gets only a third of the water: the river reaches 18 rows up the valley and
catches groundwater on its way south, and the coast drains only the southern
row.

### What the sea does not carry: the weight of salt water

The boundary above holds the coast at sea level in every layer, which is right
for a model that simulates density: the Buoyancy (**BUY**) package turns the
salinity of the boundary into a fluid density and works out the pressure at
depth for itself. This model does not simulate density, so it treats the sea as
fresh water all the way down.

A constant-density model can be given some of the missing weight by hand, as an
**equivalent freshwater head**: the head a column of fresh water would need to
balance the pressure of the salt water above it, which rises with depth. Build
the same boundary that way and run it, to see how much of the answer rests on
that choice.

In [ ]:
ghb_efh, _ = coastal_ghb_data(
    gwf, row=south, skip_cols=(outlet_col,), sea_level=0.0, equivalent_freshwater=True
)
by_layer = {}
for (k, _, _), bhead, *_ in ghb_efh:
    by_layer.setdefault(k, bhead)
for k, bhead in sorted(by_layer.items()):
    print(f"layer {k + 1}: boundary head {to_display(bhead):6.3f} {unit(plain=True)}")

efh_ws = model_root / "coastal-ghb-efh"
if efh_ws.exists():
    shutil.rmtree(efh_ws)
shutil.copytree(base_ws, efh_ws)
sim = flopy.mf6.MFSimulation.load(sim_ws=str(efh_ws), verbosity_level=0)
flopy.mf6.ModflowGwfghb(
    sim.get_model(),
    pname="ghb-1",
    auxiliary=ghb_aux,
    boundnames=True,
    stress_period_data={0: ghb_efh},
)
sim.write_simulation(silent=True)
success, buff = flopy.run_model(
    exe_name=str(mf6_exe), namefile=None, model_ws=str(efh_ws), silent=True
)
if not success:
    raise RuntimeError("\n".join(buff[-15:]))

print()
print(f"river, no coast:            {river_before:12,.0f} {flow_unit}")
print(f"river, sea level:           {river_after:12,.0f} {flow_unit}")
print(
    f"river, equivalent fw head:  {net_flow(efh_ws, 'RIV(RIV-1)'):12,.0f} {flow_unit}"
)
print(f"drains, sea level:          {drains_after:12,.0f} {flow_unit}")
print(
    f"drains, equivalent fw head: {net_flow(efh_ws, 'DRN(DRN-1)'):12,.0f} {flow_unit}"
)
print(f"coast, sea level:           {coast_after:12,.0f} {flow_unit}")
print(
    f"coast, equivalent fw head:  {net_flow(efh_ws, 'GHB(GHB-1)'):12,.0f} {flow_unit}"
)

**What to look for.** The boundary head rises from 0.019 m (0.061 ft) in layer 1
to 1.760 m (5.774 ft) in layer 5, so the deep cells are now held above the
aquifer. They stop taking water and start supplying it: the coast changes from
12,713 m³/d (448,946 ft³/d) of discharge to 1,181 m³/d (41,705 ft³/d) of
inflow, and the river ends up carrying 38,613 m³/d (1,363,589 ft³/d), more than
the 37,510 m³/d (1,324,666 ft³/d) it carried with no coast at all.

That is too much. The equivalent freshwater head assumes salt water fills the
aquifer to the bottom of layer 5, 46 to 91 m (150 to 298 ft) below sea level
along the coast, when a real coast holds a wedge of salt water that thins inland
and sits under fresh water. Neither the sea-level boundary nor this one has the
wedge in it, because a constant-density model cannot produce one. The sea-level
boundary is what the datasets carry, and the wedge is where
[`mf6-coastal-density`](mf6-coastal-density.ipynb) starts.

## The advanced valley

The advanced valley replaces the river with an SFR network, the
high-conductivity lake cells with a lake (**LAK**) package that sits on top of
the aquifer, recharge with unsaturated-zone flow (**UZF**), and the production
wells with multi-aquifer wells (**MAW**). A water mover (**MVR**) sends runoff
from the unsaturated zone and the water the seepage drains collect to the lake
or to the stream reach the land surface drains to. The same coastal boundary is
already built into the shipped coastal dataset, so load it and its no-coast twin
and run both.

In [ ]:
adv_no_coast_ws = model_root / "coastal-ghb-adv-no-coast"
adv_coastal_ws = model_root / "coastal-ghb-adv"

for ws, src in (
    (adv_no_coast_ws, data_root / "synthetic-valley-advanced-annual"),
    (adv_coastal_ws, data_root / "synthetic-valley-coastal-advanced-annual"),
):
    if ws.exists():
        shutil.rmtree(ws)
    shutil.copytree(src, ws)
    success, buff = flopy.run_model(
        exe_name=str(mf6_exe), namefile=None, model_ws=str(ws), silent=True
    )
    if not success:
        raise RuntimeError("\n".join(buff[-15:]))
print("both simulations converged")

The stream and the lake both write observation files, so read the gauge flow and
the lake stage from those rather than from the budget.

In [ ]:
def stream_gauge(ws):
    """Streamflow at the downstream gauge in the last stress period."""
    flow = -pd.read_csv(pl.Path(ws) / "sv.sfr.obs.csv").iloc[-1]["RIV-FLOW"]
    return to_display(flow, "streamflow")


def lake_stage(ws):
    """Lake stage at the end of the simulation."""
    stage = pd.read_csv(pl.Path(ws) / "sv.lake.obs.csv").iloc[-1]["LAKE-STAGE"]
    return to_display(stage)


stream_unit = unit("streamflow", plain=True)
print(f"gauge flow, no coast:   {stream_gauge(adv_no_coast_ws):12.3f} {stream_unit}")
print(f"gauge flow, coastal:    {stream_gauge(adv_coastal_ws):12.3f} {stream_unit}")
print(
    f"coast discharge:        {net_flow(adv_coastal_ws, 'GHB(GHB-1)'):12,.0f} {flow_unit}"
)
# the drains hand their water to the mover, so it is booked as DRN-TO-MVR
print(
    f"drains, no coast:       {net_flow(adv_no_coast_ws, 'DRN-TO-MVR(DRN-1)'):12,.0f} {flow_unit}"
)
print(
    f"drains, coastal:        {net_flow(adv_coastal_ws, 'DRN-TO-MVR(DRN-1)'):12,.0f} {flow_unit}"
)
print(f"lake stage, no coast:   {lake_stage(adv_no_coast_ws):12.4f} {unit(plain=True)}")
print(f"lake stage, coastal:    {lake_stage(adv_coastal_ws):12.4f} {unit(plain=True)}")

**What to look for.** The stream loses about the same share as the river did:
0.209 m³/s (7.382 ft³/s) at the gauge compared to 0.323 m³/s (11.399 ft³/s)
without the coast, or 65 percent, while the coast takes 8,997 m³/d (317,723
ft³/d). The seepage drains carry nothing in the last stress period of either
run.

The lake stage falls 0.44 m (1.45 ft), from 4.7898 to 4.3490 m (15.7145 to
14.2684 ft), even though the lake ends 23 rows from the coast. The difference is
already 0.44 m (1.43 ft) at the end of the steady-state period and stays between
0.42 and 0.46 m (1.40 and 1.48 ft) through the 20 years. In that steady state
the mover delivers 1,175 m³/d (41,488 ft³/d) to the lake without the coast and
12 m³/d (441 ft³/d) with it: the coast lowers the water table enough that the
seepage drains, which send 1,123 m³/d (39,663 ft³/d) to the mover without the
coast, stop flowing. With less water arriving, the lake settles at a lower
stage, covers less area, and catches less rain. The lake is coupled to the coast
through the drains and the mover as well as through the aquifer.

## Recap

- A GHB at sea level along the southern row turns the valley into a coastal
  aquifer. It holds that edge within 0.11 m (0.34 ft) of sea level and lowers
  layer 1 heads by 0.46 m (1.51 ft) on average.
- Conductance comes from the grid and the conductivity arrays, one value per
  cell, and the outlet column is left out so the river or stream keeps the only
  surface outflow there.
- The coast takes a third of what the river or stream and the coast together
  discharge, in both the base and the advanced valley, so the river and the
  stream keep most of their flow.
- Giving the boundary an equivalent freshwater head instead, to carry the weight
  of the salt water, reverses the flow at the coast and sends more water to the
  river than the valley had with no coast, because it assumes salt water to the
  bottom of the aquifer.
- The `ELEVATION` and `CONCENTRATION` auxiliary variables do nothing here. They
  are what the Buoyancy package reads once density varies, which happens in
  [`mf6-coastal-density`](mf6-coastal-density.ipynb).